# Used Car Price Prediction

Rusty Bargain, a used car sales service, is developing an app to attract new customers, in which users can quickly find out the market value of their car. Historical data is available with technical specifications, trim versions and prices, and the goal is to build a model that determines a car's market value.

Rusty Bargain is interested in:

- the quality of the prediction;
- the speed of the prediction;
- the time required for training.

This notebook compares four regression models — Linear Regression, Gradient Boosting, Random Forest and LightGBM — on prediction quality (RMSE and R²) and training time.

## Data Preparation

The dataset contains 354,369 listings and 16 columns. Column names are converted to lowercase. Missing values are present in `vehicletype`, `gearbox`, `model`, `fueltype` and `notrepaired`.

Columns that are not considered useful for predicting price are removed: the listing metadata (`datecrawled`, `datecreated`, `lastseen`, `numberofpictures`) and `registrationmonth`.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import time
from lightgbm import LGBMRegressor

In [2]:
df = pd.read_csv('./datasets/car_data.csv')
columns = {'DateCrawled': 'datecrawled',
           'Price': 'price',
           'VehicleType': 'vehicletype',
           'RegistrationYear': 'registrationyear',
           'Gearbox': 'gearbox',
           'Power': 'power',
           'Model': 'model',
           'Mileage': 'mileage',
           'RegistrationMonth': 'registrationmonth',
           'FuelType': 'fueltype',
           'Brand': 'brand',
           'NotRepaired': 'notrepaired',
           'DateCreated': 'datecreated',
           'NumberOfPictures': 'numberofpictures',
           'PostalCode': 'postalcode',
           'LastSeen': 'lastseen'}
df.rename(columns=columns, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   datecrawled        354369 non-null  object
 1   price              354369 non-null  int64 
 2   vehicletype        316879 non-null  object
 3   registrationyear   354369 non-null  int64 
 4   gearbox            334536 non-null  object
 5   power              354369 non-null  int64 
 6   model              334664 non-null  object
 7   mileage            354369 non-null  int64 
 8   registrationmonth  354369 non-null  int64 
 9   fueltype           321474 non-null  object
 10  brand              354369 non-null  object
 11  notrepaired        283215 non-null  object
 12  datecreated        354369 non-null  object
 13  numberofpictures   354369 non-null  int64 
 14  postalcode         354369 non-null  int64 
 15  lastseen           354369 non-null  object
dtypes: int64(7), object(

In [3]:
### Drop columns that are not useful for the model
df.drop(['datecrawled', 'registrationmonth', 'numberofpictures', 'datecreated', 'lastseen'], axis=1, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   price             354369 non-null  int64 
 1   vehicletype       316879 non-null  object
 2   registrationyear  354369 non-null  int64 
 3   gearbox           334536 non-null  object
 4   power             354369 non-null  int64 
 5   model             334664 non-null  object
 6   mileage           354369 non-null  int64 
 7   fueltype          321474 non-null  object
 8   brand             354369 non-null  object
 9   notrepaired       283215 non-null  object
 10  postalcode        354369 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 29.7+ MB


## Model Training

Categorical features are one-hot encoded with `pd.get_dummies(drop_first=True)`; missing categories are encoded as all-zero dummy columns rather than imputed. The data is split into training (75%) and validation (25%) sets. For each model, the training time is measured with `time.time()`, and RMSE and R² are calculated on the validation set.

In [4]:
features = df.drop(columns=['price'],axis=1)
features = pd.get_dummies(features, drop_first=True)
target = df['price']

features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=0.25, random_state=42)


### Linear Regression (baseline)

In [5]:
model = LinearRegression()
start = time.time()
model.fit(features_train, target_train)
train_time = time.time() - start
print(f'Training time: {train_time:.2f} seconds')

pred_valid = model.predict(features_valid)

rmse = root_mean_squared_error(target_valid, pred_valid)
r2 = r2_score(target_valid, pred_valid)
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

Training time: 1.98 seconds
RMSE: 3288.3846913854245
R2 Score: 0.47520845409421475


### Gradient Boosting Regressor (scikit-learn)

In [6]:
model = GradientBoostingRegressor(n_estimators=10, learning_rate=0.1, max_depth=3)
start = time.time()
model.fit(features_train, target_train)
train_time = time.time() - start
print(f'Training time: {train_time:.2f} seconds')

pred_valid = model.predict(features_valid)

rmse = root_mean_squared_error(target_valid, pred_valid)
r2 = r2_score(target_valid, pred_valid)
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

Training time: 8.86 seconds
RMSE: 3070.8063226543877
R2 Score: 0.5423573971835882


### Random Forest Regressor

In [14]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10, 
    random_state=42
)

start = time.time()
model.fit(features_train, target_train)
train_time = time.time() - start

print(f'Training time: {train_time:.2f} seconds')

pred_valid = model.predict(features_valid)

rmse = root_mean_squared_error(target_valid, pred_valid)
r2 = r2_score(target_valid, pred_valid)

print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

Training time: 354.07 seconds
RMSE: 2053.2937065331607
R2 Score: 0.7953914645717688


### LightGBM Regressor

In [ ]:
model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=7,   
    random_state=42
)

start = time.time()
model.fit(features_train, target_train)
train_time = time.time() - start

print(f'Training time: {train_time:.2f} seconds')

pred_valid = model.predict(features_valid)

rmse = root_mean_squared_error(target_valid, pred_valid)
r2 = r2_score(target_valid, pred_valid)

print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004501 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1199
[LightGBM] [Info] Number of data points in the train set: 265776, number of used features: 291
[LightGBM] [Info] Start training from score 4408.063264
Training time: 0.71 seconds
RMSE: 2107.6483009268622
R2 Score: 0.7844153276270452


## Model Analysis

| Model | Main hyperparameters | Training time (s) | Validation RMSE | Validation R² |
|---|---|---|---|---|
| Linear Regression | default | 1.98 | 3,288.38 | 0.475 |
| Gradient Boosting (scikit-learn) | 10 trees, depth 3, learning rate 0.1 | 8.86 | 3,070.81 | 0.542 |
| Random Forest | 200 trees, depth 10 | 354.07 | **2,053.29** | **0.795** |
| LightGBM | 200 trees, depth 3, 7 leaves, learning rate 0.05 | **0.71** | 2,107.65 | 0.784 |

- **Prediction quality:** the tree ensembles clearly outperform the linear baseline. Random Forest achieves the lowest RMSE (≈ 2,053), followed closely by LightGBM (≈ 2,108).
- **Training time:** Random Forest took about 354 seconds to train, roughly 500 times longer than LightGBM (0.71 seconds), which was the fastest model tested.
- **Trade-off:** LightGBM delivers almost the same quality as Random Forest (RMSE about 54 higher, R² 0.784 vs. 0.795) at a small fraction of the training time, which makes it the best balance between quality and training speed among the models tested.
- Prediction speed was not measured separately in this notebook.